In [ ]:
# Results are already saved by analytics.analyze()
print("Processing complete! Results saved to:")
print(f"\n  ✓ {config.get_processed_data_path('summary_statistics.csv')}")
print(f"  ✓ {config.get_processed_data_path('customer_insights.csv')}")
print(f"  ✓ {config.get_processed_data_path('product_performance.csv')}")
print(f"  ✓ {config.get_processed_data_path('order_status_distribution.csv')}")
print(f"  ✓ {config.get_processed_data_path('category_performance.csv')}")

# Create a summary report
summary_report = {
    "Report Generated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Total Customers": data['customers'].count(),
    "Total Products": data['products'].count(),
    "Total Orders": data['orders'].count(),
    "Total Order Items": data['order_items'].count(),
    "Data Location": str(config.data_dir),
    "Analysis Complete": True
}

print("\n=== PIPELINE EXECUTION SUMMARY ===")
for key, value in summary_report.items():
    print(f"  {key}: {value}")

# Close Spark session
analytics.stop()
print("\n✓ Pipeline execution complete! Spark session closed.")

## Section 7: Export Processed Results

Save the processed and analyzed data to output formats for further use or reporting.

In [ ]:
# Visualization 1: Order Status Distribution
print("Creating visualizations...\n")

status_df = results['Order Status Distribution']
fig1 = px.pie(
    status_df,
    values='count',
    names='order_status',
    title='Order Status Distribution',
    template='plotly_white'
)
fig1.show()

# Visualization 2: Top 10 Products by Revenue
top_products = results['Product Performance'].head(10)
fig2 = px.bar(
    top_products,
    x='product_name',
    y='total_revenue',
    title='Top 10 Products by Revenue',
    labels={'product_name': 'Product', 'total_revenue': 'Revenue ($)'},
    template='plotly_white'
)
fig2.update_xaxes(tickangle=-45)
fig2.show()

# Visualization 3: Category Performance
category_df = results['Category Performance']
fig3 = px.bar(
    category_df,
    x='category',
    y='total_revenue',
    title='Revenue by Product Category',
    labels={'category': 'Category', 'total_revenue': 'Revenue ($)'},
    template='plotly_white'
)
fig3.update_xaxes(tickangle=-45)
fig3.show()

# Visualization 4: Top 10 Customers by Spending
top_customers = results['Customer Insights'].head(10).copy()
top_customers['customer_label'] = top_customers['name'] + ' (' + top_customers['customer_id'].astype(str) + ')'

fig4 = px.bar(
    top_customers,
    x='customer_label',
    y='total_spent',
    title='Top 10 Customers by Total Spending',
    labels={'customer_label': 'Customer', 'total_spent': 'Total Spent ($)'},
    template='plotly_white'
)
fig4.update_xaxes(tickangle=-45)
fig4.show()

print("✓ Visualizations created successfully!")

## Section 6: Visualize Insights

Create visualizations of the analyzed results using Plotly to display key business metrics.

In [ ]:
# Run complete analysis pipeline
print("Running comprehensive analytics...")
results = analytics.analyze()

print("\n✓ Analysis completed!")

# Display summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(results['Summary Statistics'].to_string(index=False))

# Display top 10 customers by spending
print("\n=== TOP 10 CUSTOMERS BY SPENDING ===")
print(results['Customer Insights'][['customer_id', 'name', 'total_spent', 'total_orders']].head(10).to_string(index=False))

# Display top products
print("\n=== TOP 10 PRODUCTS BY REVENUE ===")
print(results['Product Performance'][['product_id', 'product_name', 'category', 'total_revenue', 'total_sold']].head(10).to_string(index=False))

# Display order status distribution
print("\n=== ORDER STATUS DISTRIBUTION ===")
print(results['Order Status Distribution'].to_string(index=False))

# Display category performance
print("\n=== CATEGORY PERFORMANCE ===")
print(results['Category Performance'].to_string(index=False))

## Section 5: Perform Business Analytics

Use PySpark SQL and DataFrame operations to calculate business insights.

In [ ]:
# Initialize PySpark analytics
analytics = SparkAnalytics(config=config)

# Load data into PySpark
print("Loading data into PySpark...")
data = analytics.load_data()

print("\n✓ Data loaded into Spark DataFrames!")
print(f"  Customers: {data['customers'].count()} rows")
print(f"  Products: {data['products'].count()} rows")
print(f"  Orders: {data['orders'].count()} rows")
print(f"  Order Items: {data['order_items'].count()} rows")

# Display sample data
print("\n--- Sample Customer Data ---")
data['customers'].limit(5).show()

print("\n--- Sample Product Data ---")
data['products'].limit(5).show()

print("\n--- Sample Order Data ---")
data['orders'].limit(5).show()

## Section 4: Load Data into PySpark

Create PySpark DataFrames from the generated data and perform initial data validation and exploration.

In [ ]:
# Initialize data generator
generator = DataGenerator(config=config)

# Generate all data
print("Generating fake e-commerce data...")
generator.generate_all()

print("\n✓ Data generation completed!")
print(f"  Customers CSV: {config.get_raw_data_path(config.customers_file)}")
print(f"  Products CSV: {config.get_raw_data_path(config.products_file)}")
print(f"  Orders CSV: {config.get_raw_data_path(config.orders_file)}")
print(f"  Order Items CSV: {config.get_raw_data_path(config.order_items_file)}")

## Section 3: Generate Fake E-commerce Data

Use Faker library to generate fake customer, product, and order datasets with proper type hints and docstrings.

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Initialize configuration
config = Config()
config.num_customers = 500
config.num_products = 100
config.num_orders = 1000

logger.info("Configuration initialized")
config.log_config()

print("\n✓ Logging and configuration setup complete!")
print(f"  Customers to generate: {config.num_customers}")
print(f"  Products to generate: {config.num_products}")
print(f"  Orders to generate: {config.num_orders}")

## Section 2: Configure Logging and Settings

Set up logging configuration and load environment settings for the data pipeline.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import libraries
import pandas as pd
import logging
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum, count, avg, max as spark_max, desc, col

# Import project modules
from src.config import Config
from src.data_generator import DataGenerator
from src.spark_analytics import SparkAnalytics

print("✓ All libraries imported successfully!")
print(f"Project root: {project_root}")

## Section 1: Import Required Libraries

Import PySpark, Pandas, Faker, logging, and visualization libraries needed for the pipeline.

# E-commerce Data Pipeline Analysis Notebook

This notebook demonstrates the complete e-commerce data pipeline:
1. Generating fake e-commerce data
2. Loading data into PySpark
3. Performing business analytics
4. Visualizing key insights
5. Exporting processed results

**Note:** Ensure all dependencies from `requirements.txt` are installed before running this notebook.